# AdaBoost for regression

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

In this notebook, we train an AdaBoost to predict the rent price of an apartment. We start with the default parameters, and then we change one component at a time, the number of trees, the loss function, the learning rate and the depth of the trees, so that we can see how each of them regulates the behaviour of the model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

### Load data

We use the apartment rent data that we prepared in the notebook of the folder prepare-data.

In [ ]:
df = pd.read_csv("../prepare-data/apartment_rent.csv", index_col="time")
df.head()

In [ ]:
# the trees from Scikit-learn take numbers, so we replace the categories by integers
categorical_vars = df.select_dtypes("str").columns
df[categorical_vars] = df[categorical_vars].apply(lambda var: var.astype("category").cat.codes)

# AdaBoost does not take missing values, so we replace them by a value outside the distribution
df = df.fillna(-1)

df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="price"),
    df["price"],
    test_size=0.3,
    random_state=0,
)

X_train.shape, X_test.shape

### Train the model with the default parameters

We show the hyperparameters with their default values, so that we can see the effect of changing them later on. When we leave the estimator empty, AdaBoost trains decision trees with a depth of three.

In [ ]:
ada = AdaBoostRegressor(
    estimator=None,
    n_estimators=50,
    learning_rate=1.0,
    loss="linear",
    random_state=0,
)

ada.fit(X_train, y_train)

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):
    for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
        rmse = root_mean_squared_error(y, model.predict(X))
        r2 = r2_score(y, model.predict(X))
        print(f"{name} rmse: {rmse:.2f}, r2: {r2:.4f}")


evaluate(ada, X_train, y_train, X_test, y_test)

In [ ]:
# the model never predicts a cheap apartment
ada.predict(X_test).min().round(2), y_test.min()

The r2 is negative, which means that this model is worse than predicting the average rent for every apartment, and the lowest prediction is far above the cheapest apartments of the test set.

The reason is that AdaBoost gives more weight to the apartments that it predicts badly, and the apartments that it predicts badly are the expensive ones, so round after round the trees chase the expensive apartments and the predictions drift upwards. Let us now change one component at a time to understand what is going on.

### The number of trees

AdaBoost adds the trees one after the other, so we can evaluate the ensemble after each new tree with `staged_predict`, and we obtain the whole curve by training the model only once.

In [ ]:
ada = AdaBoostRegressor(n_estimators=100, random_state=0)
ada.fit(X_train, y_train)

train_curve = [root_mean_squared_error(y_train, p) for p in ada.staged_predict(X_train)]
test_curve = [root_mean_squared_error(y_test, p) for p in ada.staged_predict(X_test)]

trees = range(1, len(train_curve) + 1)

plt.plot(trees, train_curve, label="train")
plt.plot(trees, test_curve, label="test")
plt.xlabel("Number of trees")
plt.ylabel("rmse")
plt.title("Performance against the number of trees")
plt.legend()
plt.show()

Here more trees make the model worse, and this is the opposite of what we saw with the random forest, where more trees never hurt. The error of the train set and the error of the test set move together, so this is not overfitting, it is the weights of the apartments becoming more and more extreme as we add trees.

### The loss function

The loss decides how much extra weight an apartment receives when we predict it badly. The linear loss raises the weight in proportion to the error, the square loss raises it with the square of the error, and the exponential loss saturates, so that the worst apartments do not take over the whole ensemble.

In [ ]:
losses = ["linear", "square", "exponential"]

curves = {}
for loss in losses:
    model = AdaBoostRegressor(n_estimators=60, loss=loss, random_state=0)
    model.fit(X_train, y_train)
    curves[loss] = [root_mean_squared_error(y_test, p) for p in model.staged_predict(X_test)]

# AdaBoost stops adding trees when a tree becomes too weak, so some curves are shorter
curves = pd.DataFrame({key: pd.Series(values) for key, values in curves.items()})
curves.index = curves.index + 1

curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse in the test set")
plt.title("Performance for different loss functions")
plt.legend(title="loss")
plt.show()

### The learning rate

The learning rate multiplies the contribution of each tree, so with a small learning rate every tree moves the ensemble a little and the weights of the apartments grow more slowly.

In [ ]:
learning_rates = [0.1, 0.5, 1.0]

curves = {}
for learning_rate in learning_rates:
    model = AdaBoostRegressor(n_estimators=60, learning_rate=learning_rate, random_state=0)
    model.fit(X_train, y_train)
    curves[learning_rate] = [root_mean_squared_error(y_test, p) for p in model.staged_predict(X_test)]

# AdaBoost stops adding trees when a tree becomes too weak, so some curves are shorter
curves = pd.DataFrame({key: pd.Series(values) for key, values in curves.items()})
curves.index = curves.index + 1

curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse in the test set")
plt.title("Performance for different learning rates")
plt.legend(title="learning rate")
plt.show()

### The depth of the trees

So far we used the trees that AdaBoost trains by default, which have a depth of three. Now we keep everything else fixed and we only change how deep each tree grows.

In [ ]:
depths = [1, 3, 6, 10, 16]

results = {}
for depth in depths:
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=depth),
        n_estimators=50,
        random_state=0,
    )
    model.fit(X_train, y_train)
    results[depth] = {
        "train": root_mean_squared_error(y_train, model.predict(X_train)),
        "test": root_mean_squared_error(y_test, model.predict(X_test)),
    }

results = pd.DataFrame(results).T
results.round(1)

In [ ]:
results.plot(marker="o")
plt.xlabel("Depth of the trees")
plt.ylabel("rmse")
plt.title("Performance against the depth of the trees")
plt.show()

The depth is by far the most important of the four components in this dataset. With shallow trees the model is useless, and with deeper trees it becomes a reasonable model, because a deeper tree predicts the expensive apartments well enough for their weights to stay under control.

From here on we work with trees of depth eight.

In [ ]:
ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=8),
    n_estimators=30,
    random_state=0,
)

ada.fit(X_train, y_train)

evaluate(ada, X_train, y_train, X_test, y_test)

### AdaBoost does not extrapolate

Every tree returns the average price of the apartments of a leaf, and AdaBoost returns one of those values, so the model can only ever answer with prices that it saw during training. To see this, we train a model with the small apartments only, and then we ask it for the price of apartments of any size.

In [ ]:
small_apartments = X_train["square_feet"] < 1200

limited = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=8),
    n_estimators=30,
    random_state=0,
)

limited.fit(X_train[small_apartments], y_train[small_apartments])

In [ ]:
# we take one apartment and change only its size
apartment = X_test.iloc[[0]]
sizes = range(400, 3200, 100)
grid = pd.concat([apartment.assign(square_feet=size) for size in sizes])

plt.plot(sizes, limited.predict(grid))
plt.axvline(1200, color="grey", linestyle="dashed")
plt.xlabel("Square feet")
plt.ylabel("Predicted rent price")
plt.title("The prediction stops growing outside the training range")
plt.show()

The dashed line marks the largest apartment that the model saw during training, and beyond that line the prediction is a flat line, exactly as with the random forest.

### How the model combines the trees

AdaBoost does not average the trees when it predicts a number. It sorts the predictions of the trees from the smallest to the largest, it adds up the weights of the trees in that order, and it returns the prediction of the first tree that takes the accumulated weight above half of the total. This is a weighted median, so the answer of the model is always the answer of one of its trees.

In [ ]:
predictions = np.array([tree.predict(apartment.to_numpy())[0] for tree in ada.estimators_])
weights = ada.estimator_weights_[:len(predictions)]

order = np.argsort(predictions)
accumulated = np.cumsum(weights[order]) / weights[order].sum()
chosen = (accumulated >= 0.5).argmax()

print(f"prediction of the model: {ada.predict(apartment)[0]:.2f}")
print(f"tree selected by the weighted median: {predictions[order][chosen]:.2f}")
print(f"average of the trees: {predictions.mean():.2f}")

In [ ]:
plt.step(predictions[order], accumulated, where="post", marker="o")
plt.axhline(0.5, color="grey", linestyle="dashed")
plt.axvline(predictions[order][chosen], color="C1", label="prediction of the model")
plt.xlabel("Prediction of each tree")
plt.ylabel("Accumulated weight")
plt.title("The prediction is the weighted median of the trees")
plt.legend()
plt.show()

In [ ]:
# because the model returns the value of one tree, the predictions are still a small set of values
len(pd.unique(ada.predict(X_test))), len(y_test)